In [ ]:
import csv
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("dark_background")

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
connection = sqlite3.connect("../db.sqlite3")
cursor = connection.cursor()

cursor.execute(
    """
    SELECT
        oi.id AS order_item_id,
        oi.quantity,
        oi.order_id,
        oi.product_id,
        p.name AS product_name,
        p.price AS product_price,
        o.datetime AS order_datetime
    FROM restaurant_orderitem oi
    JOIN restaurant_product p ON oi.product_id = p.id
    JOIN restaurant_order o ON oi.order_id = o.id
    """
)
rows = cursor.fetchall()
columns = [description[0] for description in cursor.description]
connection.close()

with open("../data.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(columns)
    writer.writerows(rows)

df = pd.read_csv("../data.csv")
print(f"Number of rows: {len(df)}")
df

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_10_by_quantity = (
    df.groupby("product_name")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)


def autopct_with_count(pct: float) -> str:
    count = round(pct / 100 * top_10_by_quantity.sum())
    return f"{pct:.1f}% ({count})"


plt.figure(figsize=(8, 8))
plt.pie(top_10_by_quantity, labels=top_10_by_quantity.index, autopct=autopct_with_count)
plt.title("Top 10 positions in menu by quantity")
plt.ylabel("Quantity")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["product_price"] * df["quantity"]

top_10_by_price = (
    df.groupby("product_name")["item_price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)


def autopct_with_amount(pct: float) -> str:
    amount = pct / 100 * top_10_by_price.sum()
    return f"{pct:.1f}% ({amount:.2f})"


plt.figure(figsize=(8, 8))
plt.pie(top_10_by_price, labels=top_10_by_price.index, autopct=autopct_with_amount)
plt.title("Top 10 positions in menu by item price")
plt.ylabel("Item Price")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["order_datetime"] = pd.to_datetime(df["order_datetime"])
df["order_hour"] = df["order_datetime"].dt.hour

income_by_hour = df.groupby("order_hour")["item_price"].sum()

plt.figure(figsize=(10, 5))
plt.bar(income_by_hour.index.astype(str), income_by_hour.values)
plt.title("Profit by Order Hour")
plt.xlabel("Order Hour")
plt.xticks(rotation=90)
plt.show()

busiest_hour = income_by_hour.idxmax()
print(f"Most profitable hour: {busiest_hour}:00, income = {income_by_hour.max():.2f}")

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df["order_day_of_week"] = df["order_datetime"].dt.day_name()

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]
income_by_day = (
    df.groupby("order_day_of_week")["item_price"]
    .sum()
    .reindex(day_order)
)

plt.figure(figsize=(10, 5))
plt.bar(income_by_day.index, income_by_day.values)
plt.title("Profit by Day of the Week")
plt.xlabel("Order Day Of The Week")
plt.xticks(rotation=45)
plt.show()

busiest_day = income_by_day.idxmax()
print(f"Most profitable day: {busiest_day}, income = {income_by_day.max():.2f}")